Created this to check if label 0 was creating issues when differentiating between classes, and wouldn´t allow improvement in accuracy
use modified_duke from duke_label_modify.py. Later eliminated all other label besides 1,2 in T staging.  

In [1]:
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRangePercentilesd, Resized,
    RandFlipd, RandRotate90d, RandGaussianNoised, EnsureTyped,
)
from sklearn.model_selection import train_test_split
import pandas as pd
from monai.data import DataLoader
import torch
from monai.networks.nets import ViT
import pydicom
pydicom.config.settings.reading_validation_mode = pydicom.config.WARN


In [2]:
df_data = pd.read_csv("./data/modified_duke.csv")
data = df_data.to_dict(orient="records")


train_data, val_data = train_test_split(
    data, test_size=0.2, stratify=[d["label"] for d in data], random_state=42
)

In [3]:

train_transforms = Compose([
    LoadImaged(keys="image", reader="ITKReader"),         
    EnsureChannelFirstd(keys="image"),                      
    Orientationd(keys="image", axcodes="RAS"),             
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0),         
             mode="bilinear"),
    ScaleIntensityRangePercentilesd(                       
        keys="image", lower=1, upper=99,
        b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),      
    RandFlipd(keys="image", prob=0.5, spatial_axis=0),     
    RandRotate90d(keys="image", prob=0.5, spatial_axes=(0, 1)),
    RandGaussianNoised(keys="image", prob=0.2),
    EnsureTyped(keys="image"),
])

val_transforms = Compose([                                  
    LoadImaged(keys="image", reader="ITKReader"),
    EnsureChannelFirstd(keys="image"),
    Orientationd(keys="image", axcodes="RAS"),
    Spacingd(keys="image", pixdim=(1.0, 1.0, 1.0), mode="bilinear"),
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99,
                                    b_min=0.0, b_max=1.0, clip=True),
    Resized(keys="image", spatial_size=(96, 96, 96)),
    EnsureTyped(keys="image"),
])


/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/transforms/io/array.py:213: UserWarning: required package for reader ITKReader is not installed, or the version doesn't match requirement.
  warnings.warn(
/home/amiyazono/miniconda3/envs/monai/lib/python3.14/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


There are many times that shows an error here as runtime error transform error. It depends mainly on num_workers (deleted and check if that resolves it) and torch.multiprocessing. 

In [4]:
from monai.data import CacheDataset
import torch.multiprocessing as mp

try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass
from sklearn.metrics import (
    cohen_kappa_score
)
train_ds = CacheDataset(data=train_data, transform=train_transforms, cache_rate=1.0)
val_ds   = CacheDataset(data=val_data,   transform=val_transforms,   cache_rate=1.0)




Loading dataset: 100%|██████████| 160/160 [03:03<00:00,  1.15s/it]


In [5]:
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=4, pin_memory=True)

from monai.networks.nets import DenseNet121
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NUM_CLASSES = 2
 
model = DenseNet121(
    spatial_dims=3,
    in_channels=1,
    out_channels=NUM_CLASSES,
    pretrained=False,
).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)


In [6]:
from sklearn.metrics import accuracy_score, confusion_matrix
num_epochs = 200
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        x = batch["image"].to(device)
        y = batch["label"].to(device)
        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()

    # validation
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch["image"].to(device)
            y = batch["label"].to(device)
            outputs = model(x)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
 
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    print(f"epoch {epoch}: val acc = {accuracy_score(all_labels, all_preds)}, KWQ = {cohen_kappa_score(all_labels, all_preds, weights="quadratic")}")

epoch 0: val acc = 0.55, KWQ = 0.08045977011494265
epoch 1: val acc = 0.5125, KWQ = 0.01015228426395931
epoch 2: val acc = 0.575, KWQ = 0.15000000000000002


epoch 3: val acc = 0.5125, KWQ = 0.0031948881789137795
epoch 4: val acc = 0.4875, KWQ = -0.03929024081115329
epoch 5: val acc = 0.575, KWQ = 0.1386953768207727
epoch 6: val acc = 0.5375, KWQ = 0.0573248407643312
epoch 7: val acc = 0.58125, KWQ = 0.15590551181102363
epoch 8: val acc = 0.56875, KWQ = 0.1246431969552807
epoch 9: val acc = 0.5875, KWQ = 0.17345021916092673
epoch 10: val acc = 0.55, KWQ = 0.09148264984227128
epoch 11: val acc = 0.525, KWQ = 0.04582548650345264
epoch 12: val acc = 0.5, KWQ = -0.008191556395715338
epoch 13: val acc = 0.5375, KWQ = 0.06210392902408113
epoch 14: val acc = 0.58125, KWQ = 0.16692570718060307
epoch 15: val acc = 0.575, KWQ = 0.14572864321608037
epoch 16: val acc = 0.5875, KWQ = 0.1729323308270676
epoch 17: val acc = 0.54375, KWQ = 0.07799179033785919
epoch 18: val acc = 0.49375, KWQ = 0.003383574284835489
epoch 19: val acc = 0.5625, KWQ = 0.12826899128268987
epoch 20: val acc = 0.53125, KWQ = 0.0430622009569378
epoch 21: val acc = 0.5375, KWQ = 0.